#  Feature Engineering
## RetailMart Inc. - Inventory Demand Forecasting

---

###  Objective
Create meaningful features from the cleaned dataset to improve model performance for predicting `units_to_stock`.

---

## 1. Import Libraries & Load Data

In [1]:
# Core libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)

print(" Libraries imported successfully!")

 Libraries imported successfully!


In [2]:
# Load cleaned dataset
df = pd.read_csv('cleaned_inventory_data.csv')

print(f" Dataset loaded!")
print(f" Shape: {df.shape}")
df.head()

 Dataset loaded!
 Shape: (47136, 55)


,transaction_id,date_id,product_id,store_id,quantity_sold,unit_price,gross_amount,discount_percentage,discount_amount,net_amount,is_promotion,current_stock_level,reorder_point,units_to_stock,stockout_flag,product_name,category,subcategory,unit_price_product,cost_price,supplier_id,shelf_life_days,weight_kg,is_perishable,store_name,store_type,city,region,store_size_sqft,opening_date,manager_id,num_employees,parking_capacity,date_display,year,month,month_name,day,day_of_week,day_name,week_of_year,quarter,season,is_weekend,is_holiday,is_month_end,supplier_name,contact_email,phone,country,lead_time_days,reliability_score,min_order_quantity,payment_terms,active_status
0,T100001,2023-01-01,P1026,ST021,2.0,82.53,536.90,0,0.0,536.90,No,282.0,31,0,0,Accessories Item 2,Electronics,Accessories,268.45,124.59,S113,NaN,2.55,No,New York Supermarket #1,Supermarket,New York,East,4893,2020-09-01,M1013,58.0,134,2023-01-01,2023.0,1.0,January,1.0,6.0,Sunday,52.0,1.0,Winter,1.0,1.0,0.0,Alliance Distribution,contact@alliancedi.com,+1-849-896-7266,USA,4.0,4.1,50,NaN,Active
1,T100002,2023-01-01,P1191,ST011,1.0,86.06,86.06,0,0.0,86.06,No,190.0,65,0,0,Fragrances Item 7,Beauty,Fragrances,86.06,39.49,S118,90.0,4.61,No,Houston Hypermarket #2,Hypermarket,Houston,South,17598,2015-07-27,M1043,77.0,376,2023-01-01,2023.0,1.0,January,1.0,6.0,Sunday,52.0,1.0,Winter,1.0,1.0,0.0,First Choice Suppliers,unknown@email.com,+1-351-989-5713,USA,28.0,3.9,10,Net 15,Active
2,T100003,2023-01-01,P1143,ST018,4.0,236.75,947.00,0,0.0,947.00,No,469.0,70,0,0,Furniture Item 7,Home & Kitchen,Furniture,236.75,134.38,S112,NaN,7.54,No,New Orleans Supermarket #3,Supermarket,New Orleans,South,5164,2016-04-29,M1030,123.0,401,2023-01-01,2023.0,1.0,January,1.0,6.0,Sunday,52.0,1.0,Winter,1.0,1.0,0.0,Trusted Vendors Inc,contact@trustedven.com,+1-790-490-2285,USA,13.0,4.6,100,Net 30,Active
3,T100004,2023-01-01,P1110,ST020,14.5,19.37,387.40,0,0.0,387.40,No,287.0,36,0,0,Frozen Item 6,Groceries,Frozen,19.37,11.57,S109,NaN,10.32,Yes,New York Warehouse #4,Warehouse,New York,East,24158,2017-10-10,M1044,40.0,319,2023-01-01,2023.0,1.0,January,1.0,6.0,Sunday,52.0,1.0,Winter,1.0,1.0,0.0,Direct Source Inc,contact@directsour.com,Not Available,Mexico,13.0,4.1,500,Net 15,Active
4,T100005,2023-01-01,P1111,ST010,14.5,23.62,448.78,0,0.0,448.78,No,172.0,28,0,0,Frozen Item 7,Grocery,Frozen,23.62,15.85,S108,180.0,2.93,Yes,Indianapolis Hypermarket #2,Hypermarket,Indianapolis,North,16910,2018-11-19,M1029,142.0,159,2023-01-01,2023.0,1.0,January,1.0,6.0,Sunday,52.0,1.0,Winter,1.0,1.0,0.0,Value Chain Partners,contact@valuechain.com,+1-962-360-6898,Canada,15.0,4.1,10,Net 15,Active


---
## 2. Date/Time Features

### 2.1 Extract Temporal Features

In [3]:
# Convert date_id to datetime
df['date_id'] = pd.to_datetime(df['date_id'])

# Extract date features (if not already present)
if 'year' not in df.columns:
    df['year'] = df['date_id'].dt.year
if 'month' not in df.columns:
    df['month'] = df['date_id'].dt.month
if 'day' not in df.columns:
    df['day'] = df['date_id'].dt.day
if 'day_of_week' not in df.columns:
    df['day_of_week'] = df['date_id'].dt.dayofweek
if 'week_of_year' not in df.columns:
    df['week_of_year'] = df['date_id'].dt.isocalendar().week.astype(int)

# Additional temporal features
df['day_of_year'] = df['date_id'].dt.dayofyear
df['is_month_start'] = (df['date_id'].dt.day == 1).astype(int)
df['is_quarter_start'] = df['date_id'].dt.is_quarter_start.astype(int)
df['is_quarter_end'] = df['date_id'].dt.is_quarter_end.astype(int)

print(" Temporal features created!")
print(f"New columns: year, month, day, day_of_week, week_of_year, day_of_year, is_month_start, is_quarter_start, is_quarter_end")

 Temporal features created!
New columns: year, month, day, day_of_week, week_of_year, day_of_year, is_month_start, is_quarter_start, is_quarter_end


### 2.2 Cyclical Encoding for Time Features

In [4]:
# Cyclical encoding for month (captures cyclical nature of months)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Cyclical encoding for day of week
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Cyclical encoding for day of year
df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

print(" Cyclical encoding applied!")
print("New columns: month_sin, month_cos, dow_sin, dow_cos, doy_sin, doy_cos")

 Cyclical encoding applied!
New columns: month_sin, month_cos, dow_sin, dow_cos, doy_sin, doy_cos


---
## 3. Lag Features & Rolling Statistics

### 3.1 Create Lag Features

In [5]:
# Sort data by product, store, and date
df = df.sort_values(['product_id', 'store_id', 'date_id']).reset_index(drop=True)

# Create lag features for quantity_sold (grouped by product and store)
for lag in [1, 7, 14, 30]:
    df[f'quantity_sold_lag_{lag}'] = df.groupby(['product_id', 'store_id'])['quantity_sold'].shift(lag)

print(" Lag features created!")
print("New columns: quantity_sold_lag_1, quantity_sold_lag_7, quantity_sold_lag_14, quantity_sold_lag_30")

 Lag features created!
New columns: quantity_sold_lag_1, quantity_sold_lag_7, quantity_sold_lag_14, quantity_sold_lag_30


### 3.2 Rolling Window Statistics

In [6]:
# Rolling statistics (7-day window)
df['quantity_rolling_mean_7'] = df.groupby(['product_id', 'store_id'])['quantity_sold'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)
df['quantity_rolling_std_7'] = df.groupby(['product_id', 'store_id'])['quantity_sold'].transform(
    lambda x: x.rolling(window=7, min_periods=1).std()
)
df['quantity_rolling_max_7'] = df.groupby(['product_id', 'store_id'])['quantity_sold'].transform(
    lambda x: x.rolling(window=7, min_periods=1).max()
)

# Rolling statistics (30-day window)
df['quantity_rolling_mean_30'] = df.groupby(['product_id', 'store_id'])['quantity_sold'].transform(
    lambda x: x.rolling(window=30, min_periods=1).mean()
)

# Fill NaN values from rolling calculations
rolling_cols = ['quantity_rolling_mean_7', 'quantity_rolling_std_7', 'quantity_rolling_max_7', 'quantity_rolling_mean_30']
for col in rolling_cols:
    df[col] = df[col].fillna(df[col].median())

print(" Rolling statistics created!")
print("New columns: quantity_rolling_mean_7, quantity_rolling_std_7, quantity_rolling_max_7, quantity_rolling_mean_30")

 Rolling statistics created!
New columns: quantity_rolling_mean_7, quantity_rolling_std_7, quantity_rolling_max_7, quantity_rolling_mean_30


---
## 4. Aggregated Features

### 4.1 Product-Level Features

In [7]:
# Average sales per product
product_stats = df.groupby('product_id').agg({
    'quantity_sold': ['mean', 'std', 'max'],
    'net_amount': 'mean'
}).reset_index()
product_stats.columns = ['product_id', 'product_avg_sales', 'product_std_sales', 
                          'product_max_sales', 'product_avg_revenue']

# Merge back
df = df.merge(product_stats, on='product_id', how='left')

print(" Product-level features created!")

 Product-level features created!


### 4.2 Store-Level Features

In [8]:
# Average sales per store
store_stats = df.groupby('store_id').agg({
    'quantity_sold': ['mean', 'std'],
    'net_amount': 'mean'
}).reset_index()
store_stats.columns = ['store_id', 'store_avg_sales', 'store_std_sales', 'store_avg_revenue']

# Merge back
df = df.merge(store_stats, on='store_id', how='left')

print(" Store-level features created!")

 Store-level features created!


### 4.3 Category-Level Features

In [9]:
# Average sales per category
if 'category' in df.columns:
    category_stats = df.groupby('category').agg({
        'quantity_sold': 'mean',
        'units_to_stock': 'mean'
    }).reset_index()
    category_stats.columns = ['category', 'category_avg_sales', 'category_avg_stock']
    
    # Merge back
    df = df.merge(category_stats, on='category', how='left')
    
    print(" Category-level features created!")

 Category-level features created!


---
## 5. Derived Features

### 5.1 Stock-Related Features

In [10]:
# Stock to reorder ratio
if 'current_stock_level' in df.columns and 'reorder_point' in df.columns:
    df['stock_to_reorder_ratio'] = df['current_stock_level'] / (df['reorder_point'] + 1)  # +1 to avoid division by zero
    df['stock_deficit'] = df['reorder_point'] - df['current_stock_level']
    df['stock_deficit'] = df['stock_deficit'].clip(lower=0)  # Only positive deficits
    df['is_below_reorder'] = (df['current_stock_level'] < df['reorder_point']).astype(int)

print(" Stock-related features created!")
print("New columns: stock_to_reorder_ratio, stock_deficit, is_below_reorder")

 Stock-related features created!
New columns: stock_to_reorder_ratio, stock_deficit, is_below_reorder


### 5.2 Price & Discount Features

In [11]:
# Price-related features
if 'unit_price' in df.columns and 'cost_price' in df.columns:
    df['profit_margin'] = (df['unit_price'] - df['cost_price']) / (df['unit_price'] + 0.01)
    df['price_to_cost_ratio'] = df['unit_price'] / (df['cost_price'] + 0.01)

# Discount features
if 'discount_percentage' in df.columns:
    df['has_discount'] = (df['discount_percentage'] > 0).astype(int)
    df['high_discount'] = (df['discount_percentage'] >= 15).astype(int)

print(" Price and discount features created!")

 Price and discount features created!


### 5.3 Supplier Features

In [12]:
# Supplier reliability features
if 'lead_time_days' in df.columns and 'reliability_score' in df.columns:
    # Lead time category
    df['lead_time_category'] = pd.cut(df['lead_time_days'], 
                                       bins=[0, 3, 7, 14, float('inf')],
                                       labels=['Fast', 'Medium', 'Slow', 'Very_Slow'])
    
    # Supplier risk (low reliability + long lead time)
    df['supplier_risk'] = (5 - df['reliability_score']) * df['lead_time_days'] / 10

print(" Supplier features created!")

 Supplier features created!


---
## 6. Interaction Features

In [13]:
# Promotion + Weekend interaction
if 'is_promotion' in df.columns and 'is_weekend' in df.columns:
    # Convert is_promotion to numeric if needed
    df['is_promotion_num'] = df['is_promotion'].map({'Yes': 1, 'No': 0, 1: 1, 0: 0}).fillna(0).astype(int)
    df['promo_weekend'] = df['is_promotion_num'] * df['is_weekend']

# Promotion + Holiday interaction
if 'is_holiday' in df.columns:
    df['promo_holiday'] = df['is_promotion_num'] * df['is_holiday']

# Season + Category interaction (will be encoded later)
if 'season' in df.columns and 'category' in df.columns:
    df['season_category'] = df['season'].astype(str) + '_' + df['category'].astype(str)

print(" Interaction features created!")
print("New columns: promo_weekend, promo_holiday, season_category")

 Interaction features created!
New columns: promo_weekend, promo_holiday, season_category


---
## 7. Encode Categorical Variables

In [14]:
# Identify categorical columns to encode
categorical_cols = ['category', 'subcategory', 'store_type', 'region', 'season', 
                    'is_perishable', 'lead_time_category', 'season_category']
categorical_cols = [col for col in categorical_cols if col in df.columns]

print(f" Categorical columns to encode: {categorical_cols}")

# Store label encoders for later use
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    # Handle NaN values
    df[col] = df[col].fillna('Unknown').astype(str)
    df[f'{col}_encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f" Encoded {col}: {len(le.classes_)} unique values")

 Categorical columns to encode: ['category', 'subcategory', 'store_type', 'region', 'season', 'is_perishable', 'lead_time_category', 'season_category']
 Encoded category: 8 unique values
 Encoded subcategory: 25 unique values
 Encoded store_type: 5 unique values
 Encoded region: 5 unique values
 Encoded season: 5 unique values
 Encoded is_perishable: 2 unique values


TypeError: Cannot setitem on a Categorical with a new category (Unknown), set the categories first

In [ ]:
# Save label encoders for Streamlit app
import joblib
joblib.dump(label_encoders, 'label_encoders.pkl')
print(" Label encoders saved to label_encoders.pkl")

---
## 8. Handle Missing Values in New Features

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print(" Columns with missing values:")
print(missing)

# Fill missing lag features with 0 or median
lag_cols = [col for col in df.columns if 'lag' in col]
for col in lag_cols:
    df[col] = df[col].fillna(0)

# Fill other numeric missing values with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("\n Missing values handled!")

---
## 9. Feature Selection

In [ ]:
# Define target variable
target = 'units_to_stock'

# Columns to exclude from features
exclude_cols = [
    'transaction_id', 'date_id', 'product_id', 'store_id', 'supplier_id',
    'product_name', 'store_name', 'supplier_name', 'manager_id',
    'date_display', 'month_name', 'day_name', 'opening_date',
    'contact_email', 'phone', 'payment_terms', 'active_status',
    'year_month', target
]

# Also exclude original categorical columns (keeping encoded versions)
exclude_cols.extend(categorical_cols)

# Get feature columns
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Keep only numeric features
feature_cols = [col for col in feature_cols if df[col].dtype in ['int64', 'float64', 'int32', 'float32']]

print(f" Total features selected: {len(feature_cols)}")
print(f"\nFeatures: {feature_cols}")

In [ ]:
# Create final features dataframe
X = df[feature_cols].copy()
y = df[target].copy()

print(f" Feature matrix X: {X.shape}")
print(f" Target vector y: {y.shape}")

---
## 10. Feature Correlation with Target

In [ ]:
# Calculate correlation with target
feature_target_corr = X.corrwith(y).abs().sort_values(ascending=False)

print(" Top 20 Features by Correlation with Target:")
print(feature_target_corr.head(20))

In [ ]:
# Visualize top features
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
top_20 = feature_target_corr.head(20)
plt.barh(range(len(top_20)), top_20.values, color='steelblue')
plt.yticks(range(len(top_20)), top_20.index)
plt.xlabel('Absolute Correlation with Target')
plt.title('Top 20 Features by Correlation with units_to_stock', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Saved: feature_importance_correlation.png")

---
## 11. Save Engineered Dataset

In [ ]:
# Save complete engineered dataset
df.to_csv('engineered_inventory_data.csv', index=False)
print(" Saved: engineered_inventory_data.csv")

# Save feature list
with open('feature_columns.txt', 'w') as f:
    for col in feature_cols:
        f.write(col + '\n')
print(" Saved: feature_columns.txt")

print(f"\n Final dataset shape: {df.shape}")
print(f" Number of features: {len(feature_cols)}")

---
## 12. Summary of Created Features

### Features Created:

| Category | Features |
|----------|----------|
| **Temporal** | year, month, day, day_of_week, week_of_year, day_of_year, is_month_start, is_quarter_start, is_quarter_end |
| **Cyclical** | month_sin, month_cos, dow_sin, dow_cos, doy_sin, doy_cos |
| **Lag** | quantity_sold_lag_1, lag_7, lag_14, lag_30 |
| **Rolling** | quantity_rolling_mean_7, rolling_std_7, rolling_max_7, rolling_mean_30 |
| **Aggregated** | product_avg_sales, store_avg_sales, category_avg_sales, etc. |
| **Stock** | stock_to_reorder_ratio, stock_deficit, is_below_reorder |
| **Price** | profit_margin, price_to_cost_ratio, has_discount, high_discount |
| **Supplier** | lead_time_category, supplier_risk |
| **Interaction** | promo_weekend, promo_holiday, season_category |
| **Encoded** | category_encoded, store_type_encoded, region_encoded, season_encoded |

---
**Next Step:** Proceed to `04_ml_modeling.ipynb` for model training